<a href="https://colab.research.google.com/github/alveana/Gait-Angle-Difference-Analysis-/blob/main/GaitAngleDifferenceAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Statistical Analysis - Gait Angle Difference Analysis (GADA)

Goal: Find which joint angles significantly changes between baseline and adjusted gait using hypothesis testing.


In [2]:
#Preprocessing Implementation that is shared between both methods
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

baseline_path = "/content/drive/MyDrive/Import Data/Gait_Event_Segmentation_Baseline.xlsx"
adjusted_path = "/content/drive/MyDrive/Import Data/Gait_Event_Segmentation_Adjusted.xlsx"

#Load files
baseline = pd.read_excel(baseline_path, engine="openpyxl")
adjusted = pd.read_excel(adjusted_path, engine="openpyxl")

#Add Condition labels
baseline["Condition"] = "baseline"
adjusted["Condition"] = "adjusted"

#Combine dataframes
df = pd.concat([baseline, adjusted], ignore_index=True)

# Save combined (can keep as Excel or CSV)
df.to_excel("gait_data_combined.xlsx", index=False)

print(df.head())





Mounted at /content/drive
   Step         Event  Event_Frame          X          Y          Z  \
0    12  Heel_Contact         1437 -80.061516 -28.320555 -12.426539   
1    13  Heel_Contact         1544 -82.645920 -26.850069 -16.088009   
2    14  Heel_Contact         1652 -79.610474 -24.707399 -11.712187   
3    15  Heel_Contact         1763 -79.927879 -28.161480 -12.293068   
4    16  Heel_Contact         1873 -75.688293 -28.699041 -12.461653   

   Set_Number Condition  
0           2  baseline  
1           2  baseline  
2           2  baseline  
3           2  baseline  
4           2  baseline  


In [3]:
step_features = (
    df.groupby(["Condition", "Set_Number"])
      .agg({
          "X": "mean",
          "Y": "mean",
          "Z": "mean"
      })
      .reset_index()
)

print(step_features.head())

  Condition  Set_Number          X          Y          Z
0  adjusted           1 -80.142532 -27.780035 -14.071649
1  adjusted           3 -78.622839 -29.120339 -13.851047
2  adjusted           5 -79.196513 -26.683024 -13.211867
3  adjusted           7 -77.344467 -27.567417 -12.764265
4  adjusted           9 -79.735609 -26.735868 -13.750222


# Event-Based Analysis


*   Structure


Goal: Focus on analyzing data at specific meaningful points in the gait cycle, rather than across the entire step.

For each row:

    * X,Y,Z -> Marker position at that event
    * Condition -> Baseline or Adjusted

Therefore, the logic focuses on the comparison between baseline vs. adjusted values.

- loads your data

- finds new event frames

- creates windows between consecutive events


In [4]:
#Event based analysis

windowed_data = []

for condition in df["Condition"].unique():

    df_cond = df[df["Condition"] == condition].reset_index(drop=True)

    # Get indices where events exist
    event_indices = df_cond[df_cond["Event"].notna()].index.tolist()

    for i in range(len(event_indices) - 1):

        start_idx = event_indices[i]
        end_idx = event_indices[i + 1]

        # Event label = first event
        event_label = df_cond.loc[start_idx, "Event"]

        # ✅ CORE CHANGE: frame window (Event A first frame → Event B first frame)
        window_df = df_cond.loc[start_idx:end_idx]

        # Compute mean inside window
        windowed_data.append({
            "Event": event_label,
            "Condition": condition,
            "X": window_df["X"].mean(),
            "Y": window_df["Y"].mean(),
            "Z": window_df["Z"].mean()
        })

# ✅ NEW dataframe (replaces raw df for analysis)
df_windowed = pd.DataFrame(windowed_data)

print("\n✅ Windowed Data Preview")
print(df_windowed.head())



✅ Windowed Data Preview
          Event Condition          X          Y          Z
0  Heel_Contact  baseline -81.353718 -27.585312 -14.257274
1  Heel_Contact  baseline -81.128197 -25.778734 -13.900098
2  Heel_Contact  baseline -79.769177 -26.434439 -12.002627
3  Heel_Contact  baseline -77.808086 -28.430261 -12.377360
4  Heel_Contact  baseline -75.665733 -28.250176 -12.882219


In [5]:
#Method 1 - Event-Based Feature Analysis

#Variables
angles = ['X', 'Y', 'Z']
events = df_windowed['Event'].unique()

#Method 1 - Event-Based Feature Analysis
results = []

def cohens_d(x, y):
    if len(x) < 2 or len(y) < 2:
        return np.nan
    return (np.mean(y) - np.mean(x)) / np.sqrt((np.std(x)**2 + np.std(y)**2)/2)

for event in events:

    # ✅ use windowed dataframe instead of raw df
    df_event = df_windowed[df_windowed['Event'] == event]

    for angle in angles:

        baseline_vals = df_event[df_event['Condition'] == 'baseline'][angle]
        adjusted_vals = df_event[df_event['Condition'] == 'adjusted'][angle]

        if len(baseline_vals) > 1 and len(adjusted_vals) > 1:

            stat, p = ttest_ind(baseline_vals, adjusted_vals, equal_var=False)

            results.append({
                "Event": event,
                "Angle_Used": angle,
                "Mean_Baseline": baseline_vals.mean(),
                "Mean_Adjusted": adjusted_vals.mean(),
                "Change": adjusted_vals.mean() - baseline_vals.mean(),
                "p_value": p,
                "Effect_Size": cohens_d(baseline_vals, adjusted_vals)
            })

results_df = pd.DataFrame(results)

# Sort by biggest change
results_df = results_df.sort_values(by="Change", key=abs, ascending=False)

# Save
results_df.to_csv("event_based_results.csv", index=False)

print("\n✅ FINAL Event-Based Analysis")
print(results_df.head())



✅ FINAL Event-Based Analysis
          Event Angle_Used  Mean_Baseline  Mean_Adjusted    Change   p_value  \
1  Heel_Contact          Y     -27.882708     -27.547415  0.335293  0.273470   
2  Heel_Contact          Z     -13.575787     -13.445952  0.129834  0.733266   
0  Heel_Contact          X     -78.908414     -78.941744 -0.033330  0.956870   

   Effect_Size  
1     0.263052  
2     0.084052  
0    -0.013038  


# Annova (Step-based Conditions) Preprocessing Code

In [6]:

import pandas as pd

# Combine datasets
df = pd.concat (
    [baseline, adjusted],
    ignore_index=True
)

# Save combined (can keep as Excel or CSV)
df.to_excel(
    "gait_data_combined.xlsx",
    index=False
)

print("Combined Dataset:")
print(df.shape)
print(df.head())

Combined Dataset:
(72, 8)
   Step         Event  Event_Frame          X          Y          Z  \
0    12  Heel_Contact         1437 -80.061516 -28.320555 -12.426539   
1    13  Heel_Contact         1544 -82.645920 -26.850069 -16.088009   
2    14  Heel_Contact         1652 -79.610474 -24.707399 -11.712187   
3    15  Heel_Contact         1763 -79.927879 -28.161480 -12.293068   
4    16  Heel_Contact         1873 -75.688293 -28.699041 -12.461653   

   Set_Number Condition  
0           2  baseline  
1           2  baseline  
2           2  baseline  
3           2  baseline  
4           2  baseline  


# ANOVA Per Event Analysis


*   Statistial Test
*   One-way ANOVA




In [7]:
from scipy.stats import f_oneway
import os

# Load combined dataset
df = pd.read_excel(
    "gait_data_combined.xlsx",
    engine="openpyxl"
)

angles = ["X", "Y", "Z"]
events = df["Event"].unique()

#Effect Size (Eta Squared)

def eta_squared(baseline_vals, adjusted_vals):
  combined = np.concatenate([baseline_vals, adjusted_vals])

  grand_mean = np.mean(combined)

  ss_between = (len(baseline_vals)*(np.mean(baseline_vals)-grand_mean)**2
                + len(adjusted_vals)*(np.mean(adjusted_vals)-grand_mean)**2)

  ss_total = np.sum((combined-grand_mean)**2)

  return ss_between/ss_total

# Event-Based ANOVA

results = []

for event in events:
  print(f"\nEvent: {event}")

  df_event = df[df["Event"] == event]

  for angle in angles:
    baseline_vals = df_event[df_event["Condition"] == "baseline"][angle]
    adjusted_vals = df_event[df_event["Condition"] == "adjusted"][angle]

    if len(baseline_vals) > 1 and len(adjusted_vals) > 1:

      F_stat, p = f_oneway(baseline_vals, adjusted_vals)
      eta = eta_squared(baseline_vals, adjusted_vals)

      results.append({
        "Event Type": event,
        "Angle Used": angle,
        "Mean Baseline": baseline_vals.mean(),
        "Mean Adjusted": adjusted_vals.mean(),
        "Change": adjusted_vals.mean() - baseline_vals.mean(),
        "F-statistic": F_stat,
        "p-value": p,
        "Eta_Squared": eta_squared(baseline_vals, adjusted_vals),
        "Effect Size": eta

      })

      #Event-Based Anova Results
      results_df = pd.DataFrame(results)

      #Sort by biggest change
      results_df = results_df.sort_values(by="Change", key=abs, ascending=False)

      #Sort by p_value in ascending order
      results_df = results_df.sort_values(by = "p-value", ascending= True)

      print("\n ANOCA Results:")
      print(results_df)


Event: Heel_Contact

 ANOCA Results:
     Event Type Angle Used  Mean Baseline  Mean Adjusted    Change  \
0  Heel_Contact          X     -78.912211     -79.008392 -0.096181   

   F-statistic   p-value  Eta_Squared  Effect Size  
0     0.016734  0.897445     0.000239     0.000239  

 ANOCA Results:
     Event Type Angle Used  Mean Baseline  Mean Adjusted    Change  \
1  Heel_Contact          Y     -27.922902     -27.577337  0.345565   
0  Heel_Contact          X     -78.912211     -79.008392 -0.096181   

   F-statistic   p-value  Eta_Squared  Effect Size  
1     0.603569  0.439837     0.008549     0.008549  
0     0.016734  0.897445     0.000239     0.000239  

 ANOCA Results:
     Event Type Angle Used  Mean Baseline  Mean Adjusted    Change  \
1  Heel_Contact          Y     -27.922902     -27.577337  0.345565   
0  Heel_Contact          X     -78.912211     -79.008392 -0.096181   
2  Heel_Contact          Z     -13.575568     -13.529810  0.045758   

   F-statistic   p-value  Eta_

#Anova Preprocessing (Helper Function)



In [8]:
def max_abs_with_direction(values):
  max_vals = np.max(values)
  min_vals = np.min(values)

  if abs(max_vals >= abs(min_vals)):
    peak = max_val
    direction = "+"
  else:
    peak = min_val
    direction = "-"

  return peak, direction

#Event-Based Maximum Absolute Value ANOVA

- First calculating the peak magnitude within each Set_Number

In [9]:
summary_rows = []

for condition in ['baseline', 'adjusted']:
  for event in events:
    event_f = df[
            (df['Condition'] == condition) &
            (df['Event'] == event)
    ]
    for set_number in event_f['Set_Number'].unique():
      subset = event_f[event_f['Set_Number'] == set_number]
      row = {
          'Condition': condition,
          'Event': event,
          'Set_Number': set_number,
      }

      for angle in angles:
        max_val = subset[angle].max()
        min_val = subset[angle].min()

        if abs(max_val) >= abs(min_val):
          row[f'{angle}_Peak'] = max_val
          row[f'{angle}_Direction'] = '+'
        else:
          row[f'{angle}_Peak'] = min_val
          row[f'{angle}_Direction'] = '-'

      summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

In [10]:
display(summary_df)

,Condition,Event,Set_Number,X_Peak,X_Direction,Y_Peak,Y_Direction,Z_Peak,Z_Direction
0,baseline,Heel_Contact,2,-82.645920,-,-28.699041,-,-16.088009,-
1,baseline,Heel_Contact,4,-83.964920,-,-32.233524,-,-16.975878,-
2,baseline,Heel_Contact,6,-83.283409,-,-29.168640,-,-17.227877,-
3,baseline,Heel_Contact,8,-84.858978,-,-30.017254,-,-17.506971,-
4,adjusted,Heel_Contact,1,-86.528603,-,-31.441133,-,-17.848001,-
5,adjusted,Heel_Contact,3,-82.329239,-,-33.428246,-,-15.545545,-
6,adjusted,Heel_Contact,5,-81.078880,-,-28.305517,-,-14.742113,-
7,adjusted,Heel_Contact,7,-82.268356,-,-30.181377,-,-15.332175,-
8,adjusted,Heel_Contact,9,-88.772125,-,-28.761930,-,-17.063543,-


#ANOVA

In [11]:
for event in events:
    print(f"\nEvent: {event}")

    event_df = summary_df[
        summary_df["Event"] == event
    ]

    for angle in angles:

        baseline_vals = event_df[
            event_df["Condition"] == "baseline"
        ][f"{angle}_Peak"]

        adjusted_vals = event_df[
            event_df["Condition"] == "adjusted"
        ][f"{angle}_Peak"]

        if len(baseline_vals) > 1 and len(adjusted_vals) > 1:

            F_stat, p = f_oneway(
                baseline_vals,
                adjusted_vals
            )
            print(f"  Angle: {angle}, F-statistic: {F_stat:.4f}, p-value: {p:.4f}")


Event: Heel_Contact
  Angle: X, F-statistic: 0.0870, p-value: 0.7766
  Angle: Y, F-statistic: 0.0976, p-value: 0.7638
  Angle: Z, F-statistic: 1.4086, p-value: 0.2740
